# Biomass Dataset EDA
Run inside the Docker container (`notebook` service) where `/data` is mounted.

In [ ]:
import sys, json
sys.path.insert(0, '/workspace')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.utils.config import load_config
from src.data.dataset import BiomassDataset, build_channel_names

cfg = load_config('/workspace/configs/unet_resnet50.yaml')
DATA_ROOT = cfg['data']['root']
print('Data root:', DATA_ROOT)

## 1. Parquet split overview

In [ ]:
split_file = cfg['data']['split_file']
df = pd.read_parquet(split_file)
print('Columns:', df.columns.tolist())
print('Shape  :', df.shape)
df.head()

In [ ]:
split_col = cfg['data'].get('split_column', 'split')
print('Split counts:')
print(df[split_col].value_counts())

## 2. Load a sample patch (no normalisation)

In [ ]:
tc_store = zarr.open(f'{DATA_ROOT}/labels/tree_count', mode='r')
mh_store = zarr.open(f'{DATA_ROOT}/labels/mean_height', mode='r')
s2_store = zarr.open(f'{DATA_ROOT}/inputs/s2_summer', mode='r')

idx = 42   # change to any patch index
tc   = np.array(tc_store[idx])
mh   = np.array(mh_store[idx])
s2   = np.array(s2_store[idx])   # [H, W, 10]

print('tree_count  – shape:', tc.shape, ' range:', np.nanmin(tc), '–', np.nanmax(tc))
print('mean_height – shape:', mh.shape, ' range:', np.nanmin(mh), '–', np.nanmax(mh))
print('s2_summer   – shape:', s2.shape)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
# S2 RGB (B4=red B3=green B2=blue → channels 2,1,0)
rgb = s2[:, :, [2, 1, 0]]
rgb = np.clip(rgb / np.nanpercentile(rgb, 98), 0, 1)
axes[0].imshow(rgb)
axes[0].set_title('S2 Summer RGB')

im1 = axes[1].imshow(tc, cmap='YlGn')
axes[1].set_title('tree_count')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(mh, cmap='viridis')
axes[2].set_title('mean_height')
plt.colorbar(im2, ax=axes[2])

valid = np.isfinite(tc) & np.isfinite(mh)
axes[3].imshow(valid, cmap='gray')
axes[3].set_title(f'Valid mask  ({valid.mean()*100:.1f}% pixels)')

plt.tight_layout()
plt.show()

## 3. Sparsity analysis across 100 random patches

In [ ]:
idx_col = cfg['data'].get('patch_idx_column', 'patch_idx')
train_indices = df.loc[df[split_col]=='train', idx_col].to_numpy()

rng = np.random.default_rng(0)
sample_idx = rng.choice(train_indices, size=min(200, len(train_indices)), replace=False)

support_ratios = []
tc_means = []
mh_means = []
for i in sample_idx:
    tc_ = np.array(tc_store[int(i)])
    mh_ = np.array(mh_store[int(i)])
    valid_ = np.isfinite(tc_) & np.isfinite(mh_)
    support_ratios.append(valid_.mean())
    if valid_.any():
        tc_means.append(tc_[valid_].mean())
        mh_means.append(mh_[valid_].mean())

print(f'Median support ratio : {np.median(support_ratios)*100:.2f}%')
print(f'Mean   support ratio : {np.mean(support_ratios)*100:.2f}%')
print(f'Patches with >0 trees: {(np.array(support_ratios)>0).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(support_ratios, bins=40, color='steelblue')
axes[0].set_xlabel('Support ratio (fraction of valid pixels)')
axes[0].set_title('Tree pixel sparsity')

axes[1].hist(tc_means, bins=40, color='green')
axes[1].set_xlabel('Mean tree_count (over valid pixels)')
axes[1].set_title('tree_count distribution')

axes[2].hist(mh_means, bins=40, color='brown')
axes[2].set_xlabel('Mean mean_height (over valid pixels)')
axes[2].set_title('mean_height distribution')

plt.tight_layout()
plt.show()

## 4. Channel names and normalisation stats

In [ ]:
channels = build_channel_names(
    cfg['data']['s1_seasons'],
    cfg['data']['s2_seasons'],
    cfg['data']['use_species'],
)
print(f'Total channels: {len(channels)}')
for i, c in enumerate(channels):
    print(f'  [{i:2d}] {c}')

In [ ]:
import pathlib
stats_path = cfg['data'].get('norm_stats_path', '/workspace/artifacts/norm_stats.json')
if pathlib.Path(stats_path).exists():
    with open(stats_path) as f:
        stats = json.load(f)
    print('mean[:8] (S1):', [round(v,4) for v in stats['mean'][:8]])
    print('std [:8] (S1):', [round(v,4) for v in stats['std'][:8]])
else:
    print('Norm stats not found – run compute_stats.py first.')